# Reproducing *Appearance-Invariant Detection of Suggestive Motion via Laban Movement Descriptors*

This notebook reproduces every number the paper draws from the **LMA motion data**. It needs
**no video and no WHAM inference** — the 61-feature Laban descriptors are precomputed and shipped
in `lma_data/`. The learned-video baseline (VideoMAE) is *not* retrained here; its per-fragment
predictions are shipped alongside so the comparison table can be rebuilt.

**What's in `lma_data/`**
| file | contents |
|---|---|
| `features.npz` | per-fragment 122-d LMA vector (mean+std of 61 descriptors), clip length, source-video id, feature names |
| `pools.json` | the candidate fragments for each task (so we can re-draw the balanced samples) and the per-tier dataset counts |
| `videomae_preds.csv` | precomputed VideoMAE per-fragment predictions (binary / three-way / four-way) |

Everything is balanced, seeded (42), and evaluated with source-video-grouped 5-fold cross-validation.
Requires only `numpy`, `pandas`, `scipy`, and `scikit-learn`.

In [1]:
import json, csv, collections
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from scipy.stats import kruskal

DATA_DIR = "lma_data"        # folder extracted from the released tar.gz
RANDOM_SEED = 42
TASKS = {"binary": "Binary", "3way_drop": "Three-way", "4way": "Four-way"}
TIERS = {0: "everyday (T0)", 1: "artistic (T1)", 2: "suggestive (T2)", 3: "explicit (T3)"}

# --- load the per-fragment LMA descriptors and their metadata ---
bundle         = np.load(f"{DATA_DIR}/features.npz", allow_pickle=True)
fragment_keys  = bundle["keys"]            # a unique id for each fragment
feature_matrix = bundle["X"]               # (n_fragments, 122) = mean+std of the 61 descriptors
clip_lengths   = bundle["length"]          # number of frames in each fragment
source_videos  = bundle["source_video"]    # the video each fragment was cut from
feature_names  = [str(name) for name in bundle["feature_names"]]

# where each fragment lives in feature_matrix, looked up by its key
row_of_key = {key: row for row, key in enumerate(fragment_keys)}

# the candidate fragments for each task, grouped by class label (before balancing)
task_pools = json.load(open(f"{DATA_DIR}/pools.json"))

# the two classifiers used everywhere (a fresh instance per call, so each fold refits cleanly)
def make_logistic_regression():
    return Pipeline([("scale", StandardScaler()),
                     ("classifier", LogisticRegression(max_iter=3000))])

def make_random_forest():
    return RandomForestClassifier(n_estimators=400, random_state=RANDOM_SEED, n_jobs=-1)

print(f"Loaded {len(fragment_keys)} fragments, {feature_matrix.shape[1]} features each.")

Loaded 7251 fragments, 122 features each.


## 1. The dataset

Four tiers of motion fragments — everyday (T0), artistic (T1), suggestive (T2), explicit (T3).
These match the *Data* paragraph of the paper: T0 1,027 / T1 1,889 / T2 1,715 / T3 2,620 fragments
(7,251 total) from 515 / 412 / 108 / 238 source videos.

In [2]:
# Per-tier fragment and source-video counts of the dataset described in the paper.
# The four-way pool lists every fragment of all four tiers; each fragment's tier is its pool
# label and its source video comes from features.npz. These are the renderable fragments the
# paper reports (Section 2, "Data"): one fragment is one tracked person in one continuous segment.
tier_of_key = {key: int(tier) for tier, keys in task_pools["4way"].items() for key in keys}

dataset = (pd.DataFrame({"key": list(tier_of_key),
                         "tier": list(tier_of_key.values()),
                         "source_video": [source_videos[row_of_key[k]] for k in tier_of_key]})
             .groupby("tier")
             .agg(fragments=("key", "size"), source_videos=("source_video", "nunique")))
dataset.index = dataset.index.map(TIERS)
dataset.loc["TOTAL"] = dataset.sum()
dataset

,fragments,source_videos
tier,,
everyday (T0),1027,515
artistic (T1),1889,412
suggestive (T2),1715,108
explicit (T3),2620,238
TOTAL,7251,1273


## 2. Building a task: balanced sampling + leak-free folds

Each task draws a class-balanced sample at seed 42 (binary caps at 1507/class; the others at the
smallest tier, 1027/class). Cross-validation folds are **grouped by source video**, so no fragment
of a video is ever split across the train and test sides of a fold.

In [3]:
def build_balanced_sample(task):
    # Draw the paper's balanced sample for a task: equal fragments per class, sampled without
    # replacement at the fixed seed. Returns features, labels, and metadata for the drawn fragments.
    pool = task_pools[task]
    per_class_cap = 2000 if task == "binary" else 1075
    sample_size = min(per_class_cap, min(len(keys) for keys in pool.values()))
    rng = np.random.RandomState(RANDOM_SEED)

    drawn_keys, labels = [], []
    for class_label, class_keys in pool.items():
        for i in rng.choice(len(class_keys), sample_size, replace=False):
            drawn_keys.append(class_keys[i])
            labels.append(int(class_label))

    rows = [row_of_key[key] for key in drawn_keys]
    return {
        "keys":          np.array(drawn_keys),
        "features":      feature_matrix[rows],
        "labels":        np.array(labels),
        "source_videos": source_videos[rows],
        "clip_lengths":  clip_lengths[rows],
    }

def out_of_fold_predictions(features, labels, source_videos, make_model, group_by_video=True):
    # One cross-validated prediction per fragment. With group_by_video, every fragment of a
    # source video stays in a single fold (leak-free); otherwise plain stratified folds.
    folds = (StratifiedGroupKFold(5, shuffle=True, random_state=RANDOM_SEED) if group_by_video
             else StratifiedKFold(5, shuffle=True, random_state=RANDOM_SEED))
    groups = source_videos if group_by_video else None
    return cross_val_predict(make_model(), features, labels, groups=groups, cv=folds, n_jobs=-1)

# pre-build each task's sample once; the rest of the notebook reuses these
samples = {task: build_balanced_sample(task) for task in TASKS}
pd.DataFrame({TASKS[task]: dict(collections.Counter(s["labels"])) for task, s in samples.items()}).fillna("-")

,Binary,Three-way,Four-way
0,1507.0,1027.0,1027
1,1507.0,-,1027
2,-,1027.0,1027
3,-,1027.0,1027


## 3. Headline classification accuracy (leak-free)

Source-video-grouped 5-fold balanced accuracy. Expected (table's upper panel): binary **0.78 / 0.81**
(logistic regression / random forest), three-way **0.71 / 0.71**, four-way **0.58 / 0.58**. Grouping
vs. plain folds should change results by only **~1 pp** (at most ~1.5 pp).

In [4]:
def accuracy(sample, make_model, group_by_video=True):
    predictions = out_of_fold_predictions(sample["features"], sample["labels"],
                                          sample["source_videos"], make_model, group_by_video)
    return balanced_accuracy_score(sample["labels"], predictions)

grouped_accuracy = pd.DataFrame({
    "LMA (logistic regression)": {TASKS[t]: accuracy(samples[t], make_logistic_regression) for t in TASKS},
    "LMA (random forest)":       {TASKS[t]: accuracy(samples[t], make_random_forest)       for t in TASKS},
}).T

ungrouped_accuracy = pd.DataFrame({
    "LMA (logistic regression)": {TASKS[t]: accuracy(samples[t], make_logistic_regression, group_by_video=False) for t in TASKS},
    "LMA (random forest)":       {TASKS[t]: accuracy(samples[t], make_random_forest,       group_by_video=False) for t in TASKS},
}).T

print(f"Largest grouped-vs-ungrouped gap: {100 * (grouped_accuracy - ungrouped_accuracy).abs().max().max():.1f} pp")
grouped_accuracy.round(3)

Largest grouped-vs-ungrouped gap: 1.3 pp


,Binary,Three-way,Four-way
LMA (logistic regression),0.776,0.706,0.575
LMA (random forest),0.814,0.711,0.579


## 4. The clip-length confound, and controlling for it

NSFW clips skew short, and short clips give degraded WHAM features — a duration shortcut. We confirm
it (clip length **alone** ≈ 0.62 on binary), then **length-match** the classes (equal counts in each
of ten clip-length deciles) so duration carries no label signal. Expected de-confounded: binary
**0.68 / 0.70**, three-way **0.64 / 0.64**; a length-only classifier on the matched subset falls to chance.

In [5]:
def length_matched_mask(labels, clip_lengths, seed):
    # Select a length-balanced subset: within each of ten clip-length deciles, keep an equal
    # number of fragments from every class, so clip length no longer predicts the label.
    edges = np.quantile(clip_lengths, np.linspace(0, 1, 11)); edges[-1] += 1
    decile = np.digitize(clip_lengths, edges[1:-1])
    rng = np.random.RandomState(seed)

    kept = []
    for d in range(10):
        rows_per_class = {label: np.where((decile == d) & (labels == label))[0] for label in np.unique(labels)}
        smallest = min(len(rows) for rows in rows_per_class.values())
        for rows in rows_per_class.values():
            kept += list(rng.choice(rows, smallest, replace=False))

    mask = np.zeros(len(labels), dtype=bool); mask[kept] = True
    return mask

def length_matched_accuracy(sample, predictions):
    # Average balanced accuracy of given predictions over three length-matched subsets.
    return np.mean([balanced_accuracy_score(sample["labels"][m], predictions[m])
                    for m in (length_matched_mask(sample["labels"], sample["clip_lengths"], s) for s in (0, 1, 2))])

# (1) clip length on its own already separates the binary classes -- this is the confound.
binary = samples["binary"]
length_feature = binary["clip_lengths"].reshape(-1, 1).astype(float)
length_only = out_of_fold_predictions(length_feature, binary["labels"], binary["source_videos"], make_logistic_regression)
print(f"binary accuracy from clip length ALONE: {balanced_accuracy_score(binary['labels'], length_only):.3f}  (the confound)")

# (2) length-match the classes, then re-measure the real classifiers (and length-only, now ~chance).
rows = {}
for task, sample in samples.items():
    logreg = out_of_fold_predictions(sample["features"], sample["labels"], sample["source_videos"], make_logistic_regression)
    forest = out_of_fold_predictions(sample["features"], sample["labels"], sample["source_videos"], make_random_forest)
    length = out_of_fold_predictions(sample["clip_lengths"].reshape(-1, 1).astype(float),
                                     sample["labels"], sample["source_videos"], make_logistic_regression)
    rows[TASKS[task]] = {
        "LMA (logistic regression)": length_matched_accuracy(sample, logreg),
        "LMA (random forest)":       length_matched_accuracy(sample, forest),
        "clip length only":          length_matched_accuracy(sample, length),
    }
matched_accuracy = pd.DataFrame(rows)   # rows = models, columns = tasks (matches grouped_accuracy)
matched_accuracy.round(3)

binary accuracy from clip length ALONE: 0.614  (the confound)


,Binary,Three-way,Four-way
LMA (logistic regression),0.676,0.629,0.516
LMA (random forest),0.713,0.630,0.528
clip length only,0.497,0.330,0.242


## 5. Comparison to the learned video model (the main table)

VideoMAE was fine-tuned on plain-mesh renders of the **same** fragments and the **same** grouped
folds; its per-fragment predictions are loaded here, not retrained. We align them to our balanced
draw and report both panels. Expected: on grouped data the LMA models lead, but once length-matched
the LMA descriptors and VideoMAE land within ~1 pp — **parity**.

In [6]:
# VideoMAE's precomputed predictions: task -> fragment key -> (true label, predicted label)
videomae = collections.defaultdict(dict)
for r in csv.DictReader(open(f"{DATA_DIR}/videomae_preds.csv")):
    videomae[r["task"]][r["key"]] = (int(r["true"]), int(r["pred"]))

def videomae_accuracy(task, length_match_seed=None):
    # VideoMAE balanced accuracy on the task's fragments; optionally on a length-matched subset.
    sample = samples[task]
    true_label = np.array([videomae[task].get(k, (-1, -1))[0] for k in sample["keys"]])
    pred_label = np.array([videomae[task].get(k, (-1, -1))[1] for k in sample["keys"]])
    keep = pred_label >= 0
    if length_match_seed is not None:
        keep = keep & length_matched_mask(sample["labels"], sample["clip_lengths"], length_match_seed)
    return balanced_accuracy_score(true_label[keep], pred_label[keep])

videomae_grouped = {TASKS[t]: videomae_accuracy(t) for t in TASKS}
videomae_matched = {TASKS[t]: np.mean([videomae_accuracy(t, s) for s in (0, 1, 2)]) for t in TASKS}

comparison = pd.concat({
    "source-video-grouped":    pd.concat([grouped_accuracy, pd.DataFrame({"VideoMAE (mesh)": videomae_grouped}).T]),
    "+ length-matched":        pd.concat([matched_accuracy.drop("clip length only"),
                                          pd.DataFrame({"VideoMAE (mesh)": videomae_matched}).T]),
})
comparison.round(3)

Binary  Three-way  Four-way
source-video-grouped LMA (logistic regression)   0.776      0.706     0.575
                     LMA (random forest)         0.814      0.711     0.579
                     VideoMAE (mesh)             0.724      0.664     0.557
+ length-matched     LMA (logistic regression)   0.676      0.629     0.516
                     LMA (random forest)         0.713      0.630     0.528
                     VideoMAE (mesh)             0.694      0.628     0.532

## 6. Effort-factor decomposition (Kruskal–Wallis H)

For each feature we run a Kruskal–Wallis test across the classes and rank features by the statistic.
Expected: **Space** (directness) is the leading *Effort* factor in the **binary** split, while
**Time** (acceleration) leads the multi-way splits at **rank 3**; all four Effort globals carry signal somewhere.

In [7]:
EFFORT_FEATURE = {"Space":  "Effort_Space_Global_mean", "Time":   "Effort_Time_Global_mean",
                  "Flow":   "Effort_Flow_Global_mean",  "Weight": "Effort_Weight_Global_mean"}

def feature_ranks_by_kruskal(sample):
    # Rank (1 = most discriminative) of every feature by its Kruskal-Wallis H across the classes.
    features, labels = sample["features"], sample["labels"]
    classes = np.unique(labels)
    h_statistic = np.zeros(features.shape[1])
    for col in range(features.shape[1]):
        if np.ptp(features[:, col]) > 0:                       # skip constant columns
            values_per_class = [features[labels == c, col] for c in classes]
            h_statistic[col] = kruskal(*values_per_class).statistic
    most_discriminative_first = np.argsort(-h_statistic)
    return {feature_names[col]: rank + 1 for rank, col in enumerate(most_discriminative_first)}

effort_ranks = pd.DataFrame({
    TASKS[task]: {factor: feature_ranks_by_kruskal(samples[task])[feature]
                  for factor, feature in EFFORT_FEATURE.items()}
    for task in TASKS
})
effort_ranks

,Binary,Three-way,Four-way
Space,10,25,34
Time,20,3,3
Flow,32,6,5
Weight,46,19,16


## 7. Directness peaks at suggestive motion

Per-tier mean joint *indirectness* (path length / net displacement; **higher = more indirect**).
Expected: non-monotonic — it peaks at **suggestive (T2)** and is **lowest at explicit (T3)**, which
is the most direct, even more so than everyday.

In [8]:
four_way = samples["4way"]
directness_cols = [feature_names.index(f"{joint}_Directness_mean")
                   for joint in ["HEAD", "PELVIS", "L_WRIST", "R_WRIST", "L_ANKLE", "R_ANKLE"]]

indirectness = {TIERS[tier]: four_way["features"][four_way["labels"] == tier][:, directness_cols].mean()
                for tier in [0, 1, 2, 3]}
pd.Series(indirectness, name="mean indirectness").round(2).to_frame()

,mean indirectness
everyday (T0),12.81
artistic (T1),13.49
suggestive (T2),15.57
explicit (T3),9.28


## 8. Four-way confusion matrix (the artistic ablation)

The four-way task is a diagnostic. Expected: the artistic↔suggestive boundary (T1↔T2) is the hardest
(~20 % mutual confusion); the everyday/explicit extremes recover at ~67–68 %; and explicit confuses
more with everyday than with suggestive.

In [9]:
four_way = samples["4way"]
predictions = out_of_fold_predictions(four_way["features"], four_way["labels"],
                                      four_way["source_videos"], make_logistic_regression)

counts = confusion_matrix(four_way["labels"], predictions)
percent = pd.DataFrame(100 * counts / counts.sum(axis=1, keepdims=True),
                       index=[TIERS[t] for t in range(4)], columns=[TIERS[t] for t in range(4)])
percent["recall"] = np.diag(percent)

print(f"artistic <-> suggestive: T1->T2 {percent.iloc[1, 2]:.0f}%, T2->T1 {percent.iloc[2, 1]:.0f}%")
print(f"explicit confuses more with everyday ({percent.iloc[3, 0]:.0f}%) than suggestive ({percent.iloc[3, 2]:.0f}%)")
percent.round(0).astype(int)

artistic <-> suggestive: T1->T2 23%, T2->T1 23%
explicit confuses more with everyday (16%) than suggestive (9%)


,everyday (T0),artistic (T1),suggestive (T2),explicit (T3),recall
everyday (T0),68,13,7,11,68
artistic (T1),17,47,23,13,47
suggestive (T2),12,23,48,17,48
explicit (T3),16,8,9,67,67


---
Every number above is regenerated from the shipped LMA descriptors and the precomputed VideoMAE
predictions — no video, no WHAM, no GPU. The only input is `lma_data/`.